# ⚽ YallaKora RAG Chatbot
**Stack:** LangChain · Groq (LLaMA 3) · Pinecone · HuggingFace Embeddings

**Pipeline:**
1. Scrape YallaKora news sitemaps
2. Preprocess & clean the Arabic/mixed text
3. Chunk → embed → upsert to Pinecone
4. Chat with a Groq-powered retrieval chain

## 1. Install Dependencies

In [1]:
!pip install -q \
    langchain langchain-community langchain-text-splitters \
    langchain-groq langchain-pinecone \
    pinecone-client \
    sentence-transformers \
    unstructured lxml \
    python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 56.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.8 MB/s eta 0:00:00


## 2. Configuration

In [20]:
import os

GROQ_API_KEY      = "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"       # from console.groq.com
PINECONE_API_KEY  = "pcsk_7WTnyf_2aJZg2NSqkX4Y5Qj5HLPzwucEyvvwbJMWANv2XQXM9kqczXf6Q7hQEoDGoghxFW"    # from app.pinecone.io → API Keys

PINECONE_INDEX    = "yallakora-rag"
PINECONE_REGION   = "us-east-1"
EMBEDDING_DIM     = 768
EMBED_MODEL       = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
GROQ_MODEL        = "openai/gpt-oss-120b"
CHUNK_SIZE        = 800
CHUNK_OVERLAP     = 100

os.environ["GROQ_API_KEY"]     = GROQ_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

# ── Quick key validation ──────────────────────────────────────
from pinecone import Pinecone
try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
    pc.list_indexes()
    print("✅ Pinecone key valid")
except Exception as e:
    print(f"❌ Pinecone auth failed: {e}")

✅ Pinecone key valid


## 3. Scrape YallaKora Sitemaps

In [4]:
import time
from langchain_community.document_loaders import UnstructuredURLLoader

# Build sitemap URL list
sitemap_urls = [
    f"https://www.yallakora.com/feed/sitemap?type=egyptnews&count=200&page={p}"
    for p in range(1, 30) # 151
] + [
    f"https://www.yallakora.com/feed/sitemap?type=internationalnews&count=200&page={p}"
    for p in range(1, 10) # 60
]

BATCH_SIZE    = 10
SLEEP_BETWEEN = 1    # seconds — be polite to the host
all_documents = []

print(f"Scraping {len(sitemap_urls)} sitemaps in batches of {BATCH_SIZE}…")

for i in range(0, len(sitemap_urls), BATCH_SIZE):
    batch = sitemap_urls[i : i + BATCH_SIZE]
    print(f"  Batch {i // BATCH_SIZE + 1}/{len(sitemap_urls) // BATCH_SIZE} …", end=" ")
    try:
        loader = UnstructuredURLLoader(
            urls=batch,
            continue_on_failure=True,
            mode="single",
            headers={"User-Agent": "Mozilla/5.0 (compatible; YallaKoraBot/1.0)"},
        )
        docs = loader.load()
        all_documents.extend(docs)
        print(f"{len(docs)} docs")
    except Exception as e:
        print(f"⚠️  failed: {e}")
    time.sleep(SLEEP_BETWEEN)

print(f"\n✅ Total documents scraped: {len(all_documents)}")

Scraping 38 sitemaps in batches of 10…
  Batch 1/3 … 10 docs
  Batch 2/3 … 10 docs
  Batch 3/3 … 10 docs
  Batch 4/3 … 8 docs

✅ Total documents scraped: 38


## 4. Preprocess & Clean

YallaKora pages contain mixed Arabic/Latin text with a lot of boilerplate (nav menus, ads, whitespace).  
We strip that noise before chunking so the embeddings stay meaningful.

In [6]:
import re
from langchain_core.documents import Document

# ── Boilerplate patterns common to YallaKora pages ────────────
BOILERPLATE_PATTERNS = [
    r"يلا كورة.*?جميع الحقوق محفوظة",   # footer copyright
    r"الصفحة الرئيسية",                  # nav: home
    r"تسجيل الدخول|إنشاء حساب",          # auth links
    r"اشترك في النشرة البريدية",          # newsletter CTA
    r"\[.*?\]",                           # leftover bracket artefacts
    r"Cookie|cookies",                    # cookie notices (English variant)
]
BOILERPLATE_RE = re.compile("|".join(BOILERPLATE_PATTERNS), re.IGNORECASE)

def clean_text(text: str) -> str:
    """Normalise and strip boilerplate from a scraped page."""
    # Collapse excessive whitespace / newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove boilerplate lines
    text = BOILERPLATE_RE.sub("", text)

    # Remove bare URLs (leave surrounding context intact)
    text = re.sub(r"https?://\S+", "", text)

    # Normalise Arabic Alef variants → bare Alef  (helps embedding consistency)
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)  # أ إ آ → ا
    text = re.sub(r"\u0649", "\u064a", text)                 # ى → ي

    return text.strip()


def is_useful(text: str, min_chars: int = 150) -> bool:
    """Reject documents that are too short or pure navigation/ads."""
    arabic_chars = len(re.findall(r"[\u0600-\u06FF]", text))
    return len(text) >= min_chars and arabic_chars >= 50


cleaned_documents = []
for doc in all_documents:
    clean = clean_text(doc.page_content)
    if is_useful(clean):
        cleaned_documents.append(
            Document(page_content=clean, metadata=doc.metadata)
        )

print(f"Before cleaning : {len(all_documents):,} documents")
print(f"After  cleaning : {len(cleaned_documents):,} documents")
print(f"Sample:\n{cleaned_documents[0].page_content[:400]}")

Before cleaning : 38 documents
After  cleaning : 38 documents
Sample:
يلا كورة 

ar

2026-05-23

مصدر يكشف ليلا كورة.. حقيقة مفاوضات الاهلي مع محمد الامين رمضاوي

محمد الامين رمضاوي , النادي الاهلي , الاهلي , رمضاوي , فريق الاهلي

hourly

1.0

2026-05-23T12:37:15+00:00



يلا كورة 

ar

2026-05-23

مصدر ليلا كورة: استاد القاهرة يستضيف ودية مصر امام روسيا 

منتخب مصر,منتخب روسيا,كاس العالم

hourly

1.0

2026-05-23T12:37:15+00:00



يلا كورة 

ar

2026-05-23

لسداد ال


## 5. Chunk Documents

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "،", " ", ""],  # Arabic-aware separators
)

chunks = splitter.split_documents(cleaned_documents)
print(f"✅ {len(chunks):,} chunks created from {len(cleaned_documents):,} documents")

✅ 2,056 chunks created from 38 documents


## 6. Embed & Index in Pinecone

In [13]:
from pinecone import Pinecone, ServerlessSpec
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

# ── Embedding model ───────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},   # change to "cuda" if you have a GPU
    encode_kwargs={"normalize_embeddings": True},
)

# ── Pinecone index ────────────────────────────────────────────
pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=PINECONE_REGION),
    )
    print(f"✅ Created Pinecone index '{PINECONE_INDEX}'")
else:
    print(f"ℹ️  Index '{PINECONE_INDEX}' already exists — reusing")

# ── Upsert in batches ─────────────────────────────────────────
UPSERT_BATCH = 100
print(f"Upserting {len(chunks):,} chunks to Pinecone…")

for i in range(0, len(chunks), UPSERT_BATCH):
    batch = chunks[i : i + UPSERT_BATCH]
    PineconeVectorStore.from_documents(
        documents=batch,
        embedding=embeddings,
        index_name=PINECONE_INDEX,
    )
    print(f"  Upserted {min(i + UPSERT_BATCH, len(chunks))}/{len(chunks)}")

print("✅ All chunks indexed!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Created Pinecone index 'yallakora-rag'
Upserting 2,056 chunks to Pinecone…
  Upserted 100/2056
  Upserted 200/2056
  Upserted 300/2056
  Upserted 400/2056
  Upserted 500/2056
  Upserted 600/2056
  Upserted 700/2056
  Upserted 800/2056
  Upserted 900/2056
  Upserted 1000/2056
  Upserted 1100/2056
  Upserted 1200/2056
  Upserted 1300/2056
  Upserted 1400/2056
  Upserted 1500/2056
  Upserted 1600/2056
  Upserted 1700/2056
  Upserted 1800/2056
  Upserted 1900/2056
  Upserted 2000/2056
  Upserted 2056/2056
✅ All chunks indexed!


## 7. Build the RAG Chain

In [21]:
from langchain_groq import ChatGroq
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# ── Load existing index (skip if already in memory) ───────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# ── Groq LLM ──────────────────────────────────────────────────
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    api_key=GROQ_API_KEY,
)

# ── Prompt ────────────────────────────────────────────────────
PROMPT_TEMPLATE = """
أنت مساعد متخصص في أخبار كرة القدم من موقع يلا كورة.
استخدم فقط المعلومات الواردة في السياق التالي للإجابة على السؤال.
إذا لم تجد الإجابة في السياق، قل "لا تتوفر لديّ هذه المعلومات".

السياق:
{context}

السؤال: {question}

الإجابة:
"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

# ── Chain ─────────────────────────────────────────────────────
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt},
)

print("✅ RAG chain ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG chain ready


## 8. Chat Interface

In [25]:
def ask(question: str, show_sources: bool = False) -> str:
    """Query the RAG chain and print the answer."""
    result = qa_chain.invoke({"query": question})
    answer = result["result"]

    print(f"\n🤖 {answer}")

    if show_sources:
        print("\n📄 Sources:")
        for doc in result["source_documents"]:
            src = doc.metadata.get("source", "unknown")
            print(f"  • {src}")

    return answer


# ── Quick test ────────────────────────────────────────────────
ask("من هو هداف الدوري المصري هذا الموسم؟", show_sources=True)


🤖 لا تتوفر لديّ هذه المعلومات.

📄 Sources:
  • https://www.yallakora.com/feed/sitemap?type=egyptnews&count=200&page=27
  • https://www.yallakora.com/feed/sitemap?type=internationalnews&count=200&page=7
  • https://www.yallakora.com/feed/sitemap?type=internationalnews&count=200&page=5
  • https://www.yallakora.com/feed/sitemap?type=internationalnews&count=200&page=9
  • https://www.yallakora.com/feed/sitemap?type=internationalnews&count=200&page=9


'لا تتوفر لديّ هذه المعلومات.'

In [ ]:
# ── Interactive loop ──────────────────────────────────────────
print("YallaKora Chatbot — اكتب 'خروج' للإنهاء\n")
while True:
    q = input("سؤالك: ").strip()
    if q.lower() in ("خروج", "exit", "quit"):
        print("مع السلامة!")
        break
    if q:
        ask(q)